# 19 — Direct Earth measurements for process matching

This notebook obtains direct Earth measurements for the top refined candidate cells. It does not substitute precipitation or temperature for minerals or ice. It queries CMR for EMIT L2A reflectance, MODIS MOD10CM snow/albedo, SMAP soil/freeze–thaw products, and NASADEM terrain. It downloads only a small candidate-area inventory and records whether measurement files are available.


In [ ]:
from pathlib import Path
import json, os, requests, pandas as pd
from IPython.display import display

ROOTS=[Path('/kaggle/working'),Path('/kaggle/input')]
def find(name):
    for root in ROOTS:
        if root.exists():
            hits=list(root.rglob(name))
            if hits:return hits[0]
    raise FileNotFoundError(name)

matches=pd.read_parquet(find('refined_earth_matches.parquet')).sort_values('rank').head(20)
OUT=Path('/kaggle/working/earth_direct_measurements'); OUT.mkdir(exist_ok=True)
token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('EARTHDATA_TOKEN')
except Exception:
    token=os.environ.get('EARTHDATA_TOKEN')
headers={'Authorization':f'Bearer {token}'} if token else {}
collections={'EMITL2ARFL':'C4079844428-LPCLOUD','MOD10CM':'C3050353608-NSIDC_CPRD','SPL3SMP':'C2938664585-NSIDC_CPRD','NASADEM_HGT':'C2763264762-LPCLOUD'}
rows=[]
for cell in matches.itertuples():
    for name,cid in collections.items():
        bbox=f'{cell.longitude-0.5},{cell.latitude-0.5},{cell.longitude+0.5},{cell.latitude+0.5}'
        q={'collection_concept_id':cid,'page_size':3,'downloadable':'true','bounding_box':bbox}
        try:
            data=requests.get('https://cmr.earthdata.nasa.gov/search/granules.json',params=q,headers=headers,timeout=45).json()
            entries=data.get('feed',{}).get('entry',[])
            for entry in entries:
                links=[x.get('href') for x in entry.get('links',[]) if x.get('href') and 'data#' in x.get('rel','')]
                rows.append({'rank':cell.rank,'latitude':cell.latitude,'longitude':cell.longitude,'product':name,'granule_id':entry.get('id'),'time_start':entry.get('time_start'),'download_url':links[0] if links else None,'metadata_only':True})
        except Exception as exc:
            rows.append({'rank':cell.rank,'latitude':cell.latitude,'longitude':cell.longitude,'product':name,'granule_id':None,'time_start':None,'download_url':None,'metadata_only':True,'error':type(exc).__name__})
manifest=pd.DataFrame(rows)
manifest.to_csv(OUT/'direct_measurement_manifest.csv',index=False)
manifest.to_parquet(OUT/'direct_measurement_manifest.parquet',index=False)
summary={'candidate_cells':int(len(matches)),'products':collections,'earthdata_token_available':bool(token),'granules_found':int(manifest.granule_id.notna().sum()),'note':'CMR granule metadata is an inventory; scientific arrays must be downloaded and decoded before process scores are computed.'}
(OUT/'manifest_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
display(manifest.groupby('product').size().rename('granules'))
print(json.dumps(summary,indent=2))
